In [0]:
client_id=dbutils.secrets.get(scope="secretscope",key="clientid")
client_secret=dbutils.secrets.get(scope="secretscope",key="secretvalue")
tenant_id=dbutils.secrets.get(scope="secretscope",key="tenantid")
temps= {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id":client_id,
        "fs.azure.account.oauth2.client.secret": client_secret,
        "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/"+tenant_id+"/oauth2/token"
    }
dbutils.fs.mount(
        source = "abfss://bronzec@prjctstrgeacc.dfs.core.windows.net/",
        mount_point = "/mnt/demo",
        extra_configs= temps
)


In [0]:
display(dbutils.fs.mounts())

In [0]:
df = spark.read.csv("/mnt/demo", header=True)
display(df)

In [0]:
df = df.dropna()
display(df)

In [0]:
df = df.withColumnRenamed("1 week % increase", "weekly_percntge_increase") \
       .withColumnRenamed("1 week change", "one_week_change") \
       .withColumnRenamed("Deaths / 100 Cases", "Death_Per_hun_cases") \
       .withColumnRenamed("Recovered / 100 Cases", "Recovered_Per_hun_cases") \
       .withColumnRenamed("Deaths / 100 Recovered", "Death_Per_hun_recovered") \
       .withColumnRenamed("Country/Region", "Country_or_Region") \
       .withColumnRenamed("Confirmed last week", "Confirmed_last_week")


In [0]:
df = df.withColumnRenamed("New cases", "New_cases") \
       .withColumnRenamed("New deaths", "New_deaths") \
       .withColumnRenamed("New recovered", "New_recovered") \
       .withColumnRenamed("WHO Region", "WHO_Region")

display(df)


In [0]:
from pyspark.sql.functions import round

df = df.withColumn("Mortality_Rate", round((df["Deaths"] / df["Confirmed"]) * 100, 2))

display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("delta").mode("overwrite").save("/mnt/silverc/cleansed_data")

In [0]:
spark.sql("create schema if not exists silver");
df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("silver.cleansed_data")

In [0]:
golddf=spark.table("silver.cleansed_data")
display(golddf) 